# Security-policy RL — live checkpoint test

Five checkpoints of a Gemma-4-26B-A4B-it run that learned a security policy with RL — which requests to **comply**
with (the policy unblocks them: answer in full) and which to **refuse** (it blocks them) — served on one endpoint
and tested against the held-out
validation set in two stages: **generate** (every prompt goes to each checkpoint) and **judge** (a judge LLM on the
same endpoint labels each response *answered / soft refusal / refused*). The results come back as precision,
recall and F1 per class in a scorecard and an F1 chart, and as the replies themselves — in the run cells and in a
gallery that follows four requests across the milestones.

The checkpoints are `avn_rl_base` (the stock model, no adapter), `avn_rl_m1`, `avn_rl_m2`, `avn_rl_m3`
(frozen early / mid / late in the run) and `avn_rl_mc` (the live run, which moves while you test); tables and
charts shorten them to `base`, `m1`, `m2`, `m3`, `mc`. Section 1 lists what each milestone is and its expected
accuracy; the run below measures each one again. Run the cells top to bottom; section 4 onwards fills in once you
have generated and judged a slice.


## 0 · Setup

The first time, a box asks for the API key you created in cynos; it is tested against the endpoint and saved to the
gitignored `.env` only if accepted, and never printed. A second box asks for the judge model (stage 2) if none is
stored. After that the cell only confirms. **Re-run this cell after any change to the repo's code** — it reloads
the modules, so no kernel restart is needed (a restart works too).


In [ ]:
import os, sys, glob, json, collections, datetime
sys.path.insert(0, os.path.abspath('..'))        # repo root
import avn_client as A, generate as G, judge as J, plots as P
import nbtheme as T
import ipywidgets as W
from IPython.display import display, HTML
import importlib
for _m in (A, P, G, J, T): importlib.reload(_m)  # re-running this cell picks up code edits; no kernel restart needed

T.inject()                                        # the stylesheet and the chart template for this page
st = A.key_setup(quiet=True)                      # tests the stored key; first time: a box that tests and saves it
def client(): return A.get_client()              # reads the stored key; never printed

data_ok = os.path.isfile(os.path.join('..', 'data', 'eval', 'policy_unblock.jsonl'))
key = {'accepted': T.pill('accepted', 'ok'), 'rejected': T.pill('rejected', 'bad'), 'missing': T.pill('not stored yet', 'warn')}[st['key']]
if st['key'] != 'accepted': key += f" <span class='small'>{T.esc(st['key_msg'])}</span>"
T.hero('Security-policy RL · live checkpoint test',
       'Five checkpoints of a Gemma-4-26B-A4B-it run that learned a security policy with RL, tested against the held-out set: generate, then judge, then read.',
       [('endpoint', f"<span class='mono'>{A.BASE_URL}</span>"),
        ('API key', key),
        ('checkpoints', ' '.join(T.pill(m, 'acc') for m in A.MODELS)),
        ('judge, stage 2', f"<span class='mono'>{T.esc(A.JUDGE_MODEL)}</span>" if A.JUDGE_MODEL else T.pill('not set — AVN_JUDGE_MODEL in .env', 'bad')),
        ('validation set', T.pill('built', 'ok') if data_ok else T.pill('not built yet', 'warn') + " <span class='small'>python ../data/build_data.py — the Run section does it for you</span>")])
for ws in st['widgets']: display(*ws)             # the key / judge boxes, only when something is missing

## 1 · The checkpoints you are about to test

Each card is a milestone: its served model name and the **training round** it was frozen at (`r0` = the stock model,
before any training), then its **expected macro F1** with the two per-class F1s underneath. Those expected numbers
were measured once on the full held-out set (221 comply + 249 refuse) and live in `checkpoints.json`. The run below
uses a small curated slice — requests the stock model gets wrong — to show what changed, so **its numbers are not
comparable to these** (section 4 says why). For comparable numbers, run the full set from the command line,
`python run_tests.py --all`, and load it in 3d. `mc` is the live run — it has no fixed number (the dashboard tracks
its target). Edit `checkpoints.json` to point a milestone at a different round.


In [ ]:
T.milestone_cards(A.CHECKPOINTS, A.reference())

## 2 · Connectivity

The endpoint has no model discovery, so this is the test: one short answer from each checkpoint, with the latency.


In [ ]:
T.connectivity(client(), A, ['avn_rl_base', 'avn_rl_m1', 'avn_rl_m2', 'avn_rl_m3', 'avn_rl_mc']);

## 3 · Run

Plain cells, no buttons. **3a** chooses a strategic slice of the held-out set. **3b** generates, one cell per
milestone, each with its own progress bar and a summary; run only the checkpoints you want to compare. **3c** judges
everything you generated and writes `results/<UTC>/` (responses, labels, metrics, report with plots). Start small:
`PER_CLASS = 6` is 12 prompts per checkpoint. `python run_tests.py --all` runs the full set from the command line.


### 3a · The slice

`choose_examples` returns a curated slice of the held-out set — not a random sample: `PER_CLASS` comply requests
and `PER_CLASS` refuse requests, **every one a request the stock model gets wrong** (a comply request it refuses,
a refuse request it answers) — the cases training had to fix, so each one has a change to show. The table is
metadata only; the request text itself appears in the run cells (3b) and the gallery (3e).


In [ ]:
PER_CLASS = 4           # requests per class for this run (comply + refuse)
SEED = 0
G.ensure_data()
HELDOUT = A.load_heldout()
CHOSEN = A.choose_examples(HELDOUT, per_class=PER_CLASS, seed=SEED)
SHOW = A.showcase_rows(HELDOUT)   # the four hand-picked requests the gallery (3e) follows across milestones; always in CHOSEN
PROMPTS = CHOSEN        # fed to generate below
GEN = {}                # rows generated in 3b, one entry per checkpoint

def run_one(model, max_tokens=2048, workers=8):
    """Stage 1 for one checkpoint: every prompt in PROMPTS, a live bar, the counts, then this checkpoint's reply to every request in the slice."""
    mid = A.model_id(model)
    bar = T.Progress(len(PROMPTS), mid)
    rows = A.generate(client(), [model], PROMPTS, max_tokens=max_tokens, workers=workers, progress=False, on_progress=bar.update)
    bar.done(); GEN[mid] = rows
    T.run_summary(mid, rows, A, showcase=SHOW)
    return rows

T.kv([('requests', f'{len(CHOSEN)} chosen — {PER_CLASS} comply + {PER_CLASS} refuse, all ones the stock model gets wrong'),
      ('showcase', f'{len(SHOW)} requests followed across the milestones in the gallery (3e)'),
      ('next', 'run the checkpoint cells below (any subset), then 3c')])
T.selection_table(CHOSEN)

### 3b · Generate — one cell per milestone

Each cell sends every request in the slice to one checkpoint and shows the counts, then **that checkpoint's reply to
every request in the slice** in a sideways scroller (comply first; the showcase ones are marked) — the same requests
in every cell, so the change is visible as you go down from `base` to `m3`, before anything is judged. Comply requests show the request and the full reply;
refuse requests show only whether the reply looks like a refusal — their text appears in the gallery (3e) once the
judge has labelled it a full refusal. The pills are a quick visible heuristic; the judge decides in 3c.


In [ ]:
run_one('avn_rl_base');

In [ ]:
run_one('avn_rl_m1');

In [ ]:
run_one('avn_rl_m2');

In [ ]:
run_one('avn_rl_m3');

In [ ]:
run_one('avn_rl_mc');

### 3c · Judge everything generated above

Stage 2: the judge LLM labels every response with the rubric our own evals use, the run is scored, and everything is
written to `results/<UTC>/`: the responses, the labels with the judge's reasons, `metrics.json`, and `report.md`
with the plots. The scorecard prints below; section 4 reads the same folder.


In [ ]:
rows_all = [r for rows in GEN.values() for r in rows]
assert rows_all, 'nothing generated yet: run at least one checkpoint cell in 3b'
out = os.path.join(os.path.abspath('../results'), datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
path = G.save_rows(rows_all, out)
json.dump({'stage': 'generate', 'endpoint': A.BASE_URL, 'models': list(GEN), 'per_class': PER_CLASS, 'seed': SEED,
           'sample': 'curated', 'n_rows': len(rows_all)}, open(os.path.join(out, 'generate.json'), 'w'), indent=1)
bar = T.Progress(len(rows_all), 'judge', unit='responses')
metrics = J.run_judge(client(), path, workers=8, on_progress=bar.update, quiet=True)
bar.done()
display(HTML(P.scorecard_html(metrics, {})))   # the curated slice is not comparable to the expected numbers (section 4)
T.note(f"Written to <span class='mono'>{os.path.relpath(out, '..')}</span>: responses.jsonl, labels.jsonl, metrics.json, report.md and plots/. "
       'Section 3d loads it into the dashboard.', 'ok')

### 3d · Load a run into the dashboard

Points the dashboard at the newest judged run on disk — the one you just made. Generate and judge a slice in
3a–3c first; re-run this cell any time to switch to the latest run.


In [ ]:
STATE = {}
def load_run():
    runs = sorted(glob.glob('../results/*/metrics.json'))
    if not runs:
        STATE.update(root=None, rows=[], metrics={}, ref={}, judge=None, sample=None); return STATE
    root = os.path.dirname(runs[-1])
    f = next((p for p in ('labels.jsonl', 'rows.jsonl') if os.path.isfile(os.path.join(root, p))), None)
    d = json.load(open(os.path.join(root, 'metrics.json')))
    STATE.update(root=root, rows=[json.loads(l) for l in open(os.path.join(root, f))] if f else [],
                 metrics=d.get('metrics', {}), ref=d.get('reference') or A.reference(), judge=d.get('judge_model'),
                 sample=(d.get('generate') or {}).get('sample'))   # 'curated' (this notebook) | 'random' | 'all' (command line)
    return STATE
load_run()
rows, metrics, ref = STATE['rows'], STATE['metrics'], STATE['ref']
if STATE['root'] is None:
    T.note('No judged run yet — generate and judge a slice in 3a–3c, then re-run this cell to load it.', 'warn')
else:
    T.kv([('dashboard source', f"<span class='mono'>{os.path.relpath(STATE['root'], '..')}</span>"),
          ('judged rows', f"{len(rows)}"), ('checkpoints', ' '.join(T.pill(m, 'acc') for m in P.model_order(metrics)) or '—'),
          ('sample', {'curated': 'curated slice (requests the stock model gets wrong): shows the change, not comparable to the expected numbers',
                      'random': 'random sample: comparable to the expected numbers',
                      'all': 'the full held-out set: comparable to the expected numbers'}.get(STATE.get('sample'), 'not recorded (an older run)')),
          ('judge', f"<span class='mono'>{T.esc(STATE['judge'] or '')}</span>")])

### 3e · Four requests, across the milestones

A hand-picked pair of legitimate requests and a pair of harmful ones, each shown across every checkpoint you
generated, so you can watch the answer change from `base` to `m3`. **Scroll each row sideways.** Legitimate
requests show the prompt and the full reply; refuse requests show the prompt too, and a reply only when the
checkpoint fully refused — a leak or a partial answer may contain harmful output, so it is withheld. Generate the
checkpoints in 3b and judge in
3c first (temperature is 1.0, so any single draw is noisy — the picks are ones that tend to change across the run).


In [ ]:
T.showcase(STATE['rows'], SHOW) if STATE.get('rows') else T.note('Generate a slice (3b) and judge it (3c), then load it in 3d — the gallery reads the judged run.', 'warn')

## 4 · Results

### Scorecard — macro F1 by checkpoint

**Macro F1 is the metric**: it rewards complying with what should be answered and refusing what should be refused
at once, so one number captures both failure modes.

**On this notebook's slice these numbers are not comparable to the expected ones** (section 1), for three reasons:
the slice is chosen from requests the stock model gets *wrong*, so `base` scores near zero by construction; it holds
only `PER_CLASS` requests per class, so one request moves recall by 100 / `PER_CLASS` points (25 at 4); and every
reply is one draw at temperature 1.0. Read it for the climb from `base` to `m3`. For numbers you can set against
the expected ones, run a random or the full sample from the command line — `python run_tests.py --per-class 50`
(about ±7 points) or `--all` (the same 470 requests, about ±3) — and load it in 3d: the scorecard then shows each
figure next to the milestone's expected number.


In [ ]:
if not metrics:
    T.note('no metrics in this run', 'warn')
elif STATE.get('sample') in ('random', 'all'):   # a random or the full sample: comparable to the expected numbers
    display(HTML(P.scorecard_html(metrics, ref)))
else:                                            # this notebook's curated slice, or an older run that did not record it
    display(HTML(P.scorecard_html(metrics, {})))
    why = ('On this curated slice (requests the stock model gets wrong) the numbers show the change from base to m3, '
           'not accuracy: they are not comparable to the expected numbers in section 1.' if STATE.get('sample') == 'curated'
           else 'This run does not record how its requests were sampled, so the expected numbers are not shown next to it.')
    T.note(why + ' For comparable numbers: <code>python run_tests.py --all</code>, then re-run 3d.', 'warn')

### F1 by checkpoint — the headline and the two classes

This run's macro F1 (left), then comply F1 (answer when it should) and refuse F1 (refuse when it should), which
show which side moved. Bars are shaded light to dark in training order: `base` lightest, `m3` the darkest frozen
milestone, `mc` (the live run) darker still. This run's numbers only; the expected ones are on the cards in section 1.


In [ ]:
P.plotly_f1(metrics).show() if metrics else T.note('no metrics', 'warn')

## 5 · Browse the judged rows

Pick a checkpoint, a class and a label. Both classes show the request; comply rows show the response, and refuse
rows show it only when it is a full refusal (a leak may contain harmful model output).


In [ ]:
def browse(model, prompt_class, label, n=8):
    target = {'comply': 'answer', 'refuse': 'refuse'}[prompt_class]
    rs = [r for r in rows if r['model'] == model and r['target'] == target and (label == 'any' or r.get('label') == label)]
    T.kv([('rows', f'{len(rs)} match, showing {min(n, len(rs))}')])
    def keep(r):
        if target == 'answer' or r.get('label') == 'full_refusal':   # comply, or a clean refusal: request and reply
            return r
        return {k: v for k, v in r.items() if k not in ('response', 'response_preview')}   # leaked reply withheld; request stays
    T.response_cards([keep(r) for r in rs[:n]], show_prompt=True, limit=900)
models = P.model_order({r['model']: 1 for r in rows}) or ['base']
W.interact(browse, model=models, prompt_class=['comply', 'refuse'],
           label=['any', 'full_compliance', 'partial_refusal', 'full_refusal'], n=W.IntSlider(8, 1, 40));

## 6 · Try your own prompt on every checkpoint

An in-scope request (authorized, your own assets, or conceptual). One sample per checkpoint at the trained settings,
side by side. The label pill is the quick visible heuristic, not the judge. Edit `MY_PROMPT` and `TRY_ON`, run the cell.


In [ ]:
MY_PROMPT = 'In two sentences, explain Kerberoasting and how defenders detect it.'
TRY_ON = ['avn_rl_base', 'avn_rl_m3']

cards = ''
for m in TRY_ON:
    fr, text = A.ask(client(), m, MY_PROMPT, max_tokens=400)
    cards += T.response_html({'id': MY_PROMPT[:60], 'model': A.short(A.model_id(m)), 'response': text, 'finish': fr},
                             show_prompt=False, heuristic=A.classify(text), limit=900)
T.h(f"<div class='cols'>{cards}</div>")

## 7 · Re-judge one response

Pick a row from the loaded run (rows that carry the full prompt and response); the judge LLM labels it again, so you
can see how stable a verdict is.


In [ ]:
candidates = [r for r in rows if r.get('response') and r.get('prompt')]
ROW = 0                 # index into candidates - change and re-run
if not candidates:
    T.note('the loaded run stores no full prompt + response rows (the committed example is sanitized); run 3b-3d first', 'warn')
else:
    r = candidates[ROW]
    verdict = A.judge(client(), r['prompt'], r['response'])
    T.kv([('row', f"{ROW} of {len(candidates)} — <span class='mono'>{T.esc(r['model'])} · {T.esc(r['id'])}</span>"),
          ('stored label', T.label_pill(r.get('label')) + f" <span class='small'>{T.esc(r.get('reason', ''))}</span>"),
          ('judge now', T.label_pill(verdict.get('label')) + f" <span class='small'>{T.esc(verdict.get('reason', ''))}</span>")])
    T.response_cards([dict(r, label=None, reason=None)], show_prompt=True, limit=1200)

---
*Temperature is 1.0, so every run is one draw: a rate measured on n prompts per class moves by about ±100·√(p(1−p)/n)
points from sampling alone (±7 at n = 50). Compare checkpoints on the same run, same prompts; use `--all` for numbers to quote.*
